# 01 · Descarga y normalización del microdato de delitos de alto impacto (SDSCJ)
**Proyecto:** narrativa visual interactiva con Plotly sobre la violencia intrafamiliar en Bogotá · Visualización de Datos, Maestría en Ciencia de Datos (Universidad Santo Tomás)
**Integrantes:** Adrian Felipe Grateron, Jhonatan Smith Doncel y Jorge Emmanuel Marin

**En una frase:** este cuaderno descarga la base oficial de delitos de Bogotá y la deja limpia y verificada antes de hacer cualquier gráfico.

**Detalle técnico:** cubre los pasos 2 y 3 de la guía del proyecto. El código reutilizable está en `src/`; aquí se ejecuta paso a paso y se explica cada decisión para que cualquiera pueda revisarla.

| Paso | Qué se hace | Módulo |
|---|---|---|
| 0 | Instalar dependencias | `requirements.txt` |
| 1 | Descargar el CSV oficial del Observatorio (botón gris *Descarga microdato de seguimiento a delitos de alto impacto*) | `src/descarga.py` |
| 2 | Leer, separar metadatos, normalizar etiquetas y tipos | `src/normalizacion.py` |
| 3 | Validar integridad (7 pruebas) y generar reporte de calidad | `src/normalizacion.py` |
| 4 | Guardar tabla tidy + diccionario de datos | `data/processed/`, `docs/` |

> **Fuente:** Secretaría Distrital de Seguridad, Convivencia y Justicia de Bogotá (SDSCJ), Observatorio de Seguridad, Convivencia y Justicia.
> URL del archivo: `https://oaiee.scj.gov.co/DatosDescarga/SIEDCO.csv` (nombre técnico del servidor; en el proyecto se guarda como `data/raw/delitos_alto_impacto_SDSCJ.csv`).

## Paso 0 · Dependencias
Instala las librerías del proyecto en el entorno (kernel) seleccionado en VS Code. Basta con hacerlo una vez; si ya están instaladas, la celda termina en segundos.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Configuración del entorno: el notebook vive en /notebooks, el código en /src
import sys, pathlib
RAIZ = pathlib.Path.cwd().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Raíz del proyecto:", RAIZ)

Raíz del proyecto: C:\Users\ADRIAN\Desktop\Ciencia de datos\Visualizacion de datos\Proyecto_Plotly_SDSCJ


## Paso 1 · Descarga con trazabilidad
**En una frase:** guardamos el archivo original tal como llega y dejamos un registro que permite comprobar que es el mismo.

`descargar()` guarda el CSV **sin modificarlo** en `data/raw/delitos_alto_impacto_SDSCJ.csv` y escribe `metadatos_descarga.json` con la fecha de descarga, el tamaño, la huella **SHA-256** y la fecha *Last-Modified* del servidor. Con la huella, cualquier evaluador puede verificar que trabaja con exactamente el mismo archivo.

**Reproducibilidad:** el ZIP de entrega ya trae `delitos_alto_impacto_SDSCJ.csv` (datos hasta el 30/09/2026), así que esta celda no descarga nada: reutiliza el archivo y comprueba que su huella coincide con la registrada. Esto importa porque el Observatorio actualiza el archivo cada mes y una descarga nueva daría otras cifras. `forzar=True` descarga la versión vigente, sabiendo que las cifras cambiarán.

In [3]:
from src import config
from src.descarga import descargar

meta = descargar(forzar=False)
meta

✔ Ya existe delitos_alto_impacto_SDSCJ.csv (33.4 MB). Usa forzar=True para volver a descargar.
✔ Huella SHA-256 verificada (b9853b1ee2ce…): es el mismo archivo del análisis.


{'url': 'https://oaiee.scj.gov.co/DatosDescarga/SIEDCO.csv',
 'archivo': 'data\\raw\\delitos_alto_impacto_SDSCJ.csv',
 'fecha_descarga': '2026-11-08T17:41:38',
 'bytes': 33390232,
 'sha256': 'b9853b1ee2ce50a294e1c271c550dbaee86ac8057b5bd1d0d1d6c75e7a575493',
 'last_modified_servidor': 'Mon, 19 Oct 2026 23:14:22 GMT'}

## Paso 2 · Inspección del archivo crudo
**En una frase:** antes de transformar los datos, los miramos tal como vienen para saber qué hay que corregir.

Lo que encontramos y que motiva la normalización:

* separador `;` y marca BOM UTF-8 al inicio;
* etiquetas en MAYÚSCULAS, poco legibles para una audiencia no técnica;
* dos columnas de **metadatos** (`Fecha de consulta`, `Datos actualizados hasta`) que solo vienen llenas en la primera fila;
* 43 categorías de arma, muchas con muy pocos casos;
* la tabla ya viene **agregada**: `CANTIDAD` es el número de casos de cada combinación de variables.

In [4]:
import pandas as pd
from src import normalizacion as N

crudo, meta_fuente = N.leer_crudo()
print("Filas:", f"{len(crudo):,}".replace(",", "."), "| Columnas:", list(crudo.columns))
print("Metadatos de la fuente:", meta_fuente)
crudo.head()

Filas: 376.185 | Columnas: ['SEXO', 'ANIO', 'MES', 'HECHO', 'LOCALIDAD', 'COD_LOCALIDAD', 'ARMA_EMPLEADA', 'RANGO_VITAL', 'RANGO_DEL_DIA', 'CANTIDAD']
Metadatos de la fuente: {'Fecha de consulta': '17/10/2026', 'Datos actualizados hasta': '30/09/2026'}


,SEXO,ANIO,MES,HECHO,LOCALIDAD,COD_LOCALIDAD,ARMA_EMPLEADA,RANGO_VITAL,RANGO_DEL_DIA,CANTIDAD
0,FEMENINO,2026,1,HURTO A PERSONAS,USAQUÉN,1,SIN EMPLEO DE ARMAS,NO REPORTADO,NOCHE,2
1,MASCULINO,2026,1,HURTO A PERSONAS,BARRIOS UNIDOS,12,SIN EMPLEO DE ARMAS,VEJEZ,NOCHE,5
2,MASCULINO,2026,1,HURTO A PERSONAS,CIUDAD BOLÍVAR,19,ARMA BLANCA / CORTOPUNZANTE,ADULTEZ,NOCHE,14
3,FEMENINO,2026,1,HURTO A PERSONAS,KENNEDY,8,SIN EMPLEO DE ARMAS,ADULTEZ,MAÑANA,92
4,MASCULINO,2026,1,HURTO A PERSONAS,BOSA,7,ARMA BLANCA / CORTOPUNZANTE,ADULTEZ,TARDE,10


In [5]:
# Cardinalidad de cada variable categórica en el crudo
crudo.drop(columns=["CANTIDAD"]).nunique().to_frame("valores_distintos")

,valores_distintos
SEXO,3
ANIO,9
MES,12
HECHO,12
LOCALIDAD,21
COD_LOCALIDAD,21
ARMA_EMPLEADA,43
RANGO_VITAL,5
RANGO_DEL_DIA,4


## Paso 3 · Normalización
**En una frase:** convertimos la tabla en una versión legible y con tipos correctos, sin cambiar ningún conteo.

`N.normalizar()` aplica estas transformaciones (cada una está comentada en `src/normalizacion.py`):

1. **Nombres de columna** en minúscula y en español claro (`HECHO` → `delito`, `RANGO_DEL_DIA` → `franja_dia`, `CANTIDAD` → `casos`).
2. **Texto:** se quitan espacios sobrantes, se normaliza Unicode (NFC) y se conservan las tildes.
3. **Etiquetas legibles:** `VIOLENCIA INTRAFAMILIAR` → *Violencia intrafamiliar*, `FEMENINO` → *Mujer*, `CIUDAD BOLÍVAR` → *Ciudad Bolívar*.
4. **Tipos:** año, mes, código y casos como enteros; se crea `fecha` (primer día de cada mes).
5. **Categorías ordenadas:** ciclo vital (infancia → vejez) y franja del día en orden cronológico (madrugada → noche).
6. **Arma:** se conserva la categoría original y se crea `arma_grupo`, que agrupa las 43 categorías en 9 grupos interpretables.

In [6]:
df = N.normalizar(crudo)
df.head()

,fecha,anio,mes,delito,cod_localidad,localidad,sexo,ciclo_vital,franja_dia,arma_grupo,arma_original,casos
0,2018-01-01,2018,1,Delitos sexuales,1,Usaquén,Mujer,Adolescencia,Tarde,Sin empleo de armas,SIN EMPLEO DE ARMAS,1
1,2018-01-01,2018,1,Delitos sexuales,1,Usaquén,Hombre,Infancia,Madrugada,Sin empleo de armas,SIN EMPLEO DE ARMAS,2
2,2018-01-01,2018,1,Delitos sexuales,1,Usaquén,Mujer,Adultez,Tarde,No reportado / por establecer,NO REPORTADO,1
3,2018-01-01,2018,1,Delitos sexuales,1,Usaquén,Mujer,Adultez,Madrugada,Sin empleo de armas,SIN EMPLEO DE ARMAS,4
4,2018-01-01,2018,1,Delitos sexuales,1,Usaquén,Hombre,Adolescencia,Madrugada,Sin empleo de armas,SIN EMPLEO DE ARMAS,1


In [7]:
df.dtypes.to_frame('tipo')

,tipo
fecha,datetime64[us]
anio,int16
mes,int8
delito,str
cod_localidad,int16
localidad,str
sexo,str
ciclo_vital,category
franja_dia,category
arma_grupo,str


## Paso 4 · Validación de integridad
**En una frase:** comprobamos automáticamente que la limpieza no perdió ni duplicó casos; si algo falla, el cuaderno se detiene.

`N.validar()` ejecuta siete pruebas: total de casos conservado · mismo número de filas · sin nulos · conteos positivos · meses entre 1 y 12 · sin duplicados en la llave tidy · correspondencia 1:1 entre código y localidad. Así no se grafica sobre datos dudosos.

También genera una **alerta de completitud del último mes**. El último mes publicado suele quedar incompleto porque las denuncias tardan en registrarse; por eso en los gráficos se marca como preliminar (punto hueco).

In [8]:
import json
reporte = N.validar(crudo, df, meta_fuente)
print(json.dumps(reporte, indent=2, ensure_ascii=False))

{
  "filas": 376185,
  "casos_totales": 1934083,
  "periodo": [
    "2018-01-01",
    "2026-09-01"
  ],
  "fecha_consulta_fuente": "17/10/2026",
  "datos_actualizados_hasta": "30/09/2026",
  "delitos": [
    "Delitos sexuales",
    "Extorsión",
    "Homicidios",
    "Hurto a comercio",
    "Hurto a entidades financieras",
    "Hurto a personas",
    "Hurto a residencias",
    "Hurto de automotores",
    "Hurto de motocicletas",
    "Lesiones personales",
    "Secuestro",
    "Violencia intrafamiliar"
  ],
  "localidades": 21,
  "categorias_arma_originales": 43,
  "grupos_arma": [
    "Arma blanca / cortopunzante",
    "Arma de fuego",
    "Contundentes",
    "Herramientas (llave maestra, palancas)",
    "Medios telefónicos / digitales",
    "No reportado / por establecer",
    "Otros medios",
    "Sin empleo de armas",
    "Sustancias (escopolamina, químicos)"
  ],
  "validaciones_superadas": [
    "Total de casos conservado",
    "Mismo número de filas",
    "Sin nulos",
    "Conteos 

## Paso 5 · Guardar la tabla limpia, el reporte de calidad y el diccionario de datos

In [9]:
df_final, reporte = N.ejecutar()   # re-ejecuta todo el flujo y escribe los archivos
for ruta in [config.ARCHIVO_NORMALIZADO, config.ARCHIVO_CALIDAD, config.ARCHIVO_DICCIONARIO]:
    print(f"✔ {ruta.relative_to(config.RAIZ)}  ({ruta.stat().st_size/1e6:.2f} MB)")

✔ Normalizado: 376.185 filas · 1.934.083 casos · 2018-01-01 → 2026-09-01
  Alerta último mes: El último mes publicado está por debajo del promedio reciente: probable rezago de registro. Se marca como PRELIMINAR en los gráficos.
✔ data\processed\delitos_alto_impacto_normalizado.csv.gz  (1.95 MB)
✔ data\processed\reporte_calidad.json  (0.00 MB)
✔ docs\diccionario_datos.md  (0.00 MB)


In [10]:
# Vista rápida: casos por delito en todo el periodo (control de cordura)
(df_final.groupby("delito")["casos"].sum().sort_values(ascending=False)
   .map(lambda v: f"{v:,.0f}".replace(",", ".")).to_frame("casos"))

,casos
delito,
Hurto a personas,1.061.471
Violencia intrafamiliar,348.013
Lesiones personales,187.291
Hurto a comercio,109.454
Hurto a residencias,69.059
Delitos sexuales,61.492
Hurto de motocicletas,40.881
Hurto de automotores,32.319
Extorsión,14.079


### Resultado
La tabla `data/processed/delitos_alto_impacto_normalizado.csv.gz` queda en formato *tidy*: cada fila es una combinación única de variables con su número de casos, con tipos correctos y etiquetas en español legible. Siguiente paso: **`02_exploracion_plotly_express.ipynb`**.